In [26]:
import torch, tqdm, time, os, json
import torch.nn as nn
import torch.nn.functional as F
from torchmetrics import Accuracy
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import MNIST
from torchvision import transforms
import lightning.pytorch as pl
from lightning.pytorch.callbacks import DeviceStatsMonitor, EarlyStopping
from pytorch_lightning.utilities.model_summary import ModelSummary
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [16]:
monitor = DeviceStatsMonitor()
monitor

In [ ]:
# set train setting
save_model = True
train_num_epoch = 50000
min_loss = 100
dl_workers = 0

In [20]:
gpu = torch.device('cuda')
torch.set_float32_matmul_precision('high')

In [18]:
'''
# load hyperparameter of json file
with open('.' + os.sep + os.path.join('models', 'params_dnn_20220207-012403'), 'r') as file:
    hyper_params = json.load(file)

s1_neuron = hyper_params['s1_neuron']
s2_neuron = hyper_params['s2_neuron']
'''

In [63]:
class GroundPressureModel(pl.LightningModule):
    def __init__(self, s1_neuron, s2_neuron):
        super().__init__()
        n_input = 4
        n_output = 4
        s1_neuron = int(round(s1_neuron))
        s2_neuron = int(round(s2_neuron))

        input_layer = [nn.Linear(n_input, s1_neuron), nn.ReLU()]
        hidden_layer = [nn.Linear(s1_neuron, s2_neuron[1]), nn.ReLU()]
        output_layer = nn.Linear(s2_neuron[1], n_output)

        self.model = nn.Sequential(*input_layer, *hidden_layer, output_layer)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self.model(x)
        loss = F.cross_entropy(y_hat, y)

        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        val_pred = self.model(x)
        val_loss = F.cross_entropy(val_pred, y)
        self.log("train_loss", val_loss)
        print(self.current_epoch, val_loss.to('cpu').detach().numpy())

        return val_loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        test_pred = self.model(x)
        test_loss = F.cross_entropy(test_pred, y)
        print(test_loss)
        self.log("test_loss", test_loss)

        return test_loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=0.02)
        return optimizer

model = GroundPressureModel(10, 10)

In [64]:
dataset = MNIST(os.getcwd(), train=True, download=True, transform=transforms.ToTensor())
train_dataset, val_dataset = random_split(dataset, [55000, 5000])
test_dataset = MNIST('', train=False, download=True, transform=transforms.ToTensor())

train_loader = DataLoader(train_dataset, batch_size=55000, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=5000, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=10000)

In [65]:
# train model
early_stop_callback = EarlyStopping(monitor='train_loss', mode='min', min_delta=0.01, patience=3, verbose=True)
trainer = pl.Trainer(callbacks=[early_stop_callback], accelerator='gpu')

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [66]:
trainer.fit(model=model, train_dataloaders=train_loader, val_dataloaders=val_loader)

You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type       | Params
-------------------------------------
0 | model | Sequential | 552 K 
-------------------------------------
552 K     Trainable params
0         Non-trainable params
552 K     Total params
2.208     Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

0 2.3023362


Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

39 0.11877501


Validation: 0it [00:00, ?it/s]

Monitored metric train_loss did not improve in the last 3 records. Best score: 0.126. Signaling Trainer to stop.


40 0.116100974


In [67]:
trainer.test(dataloaders=test_loader)

You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
Restoring states from the checkpoint path at D:\Workspace\Python\estimate_crawler_crane_ground_pressure\lightning_logs\version_14\checkpoints\epoch=40-step=41.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at D:\Workspace\Python\estimate_crawler_crane_ground_pressure\lightning_logs\version_14\checkpoints\epoch=40-step=41.ckpt


Testing: 0it [00:00, ?it/s]

tensor(0.1245, device='cuda:0')


[{}]